In [ ]:
import sys, subprocess

def _pip(args):
    subprocess.run([sys.executable, "-m", "pip"] + args, check=False)

# 1. Uninstall every onnxruntime / rapidocr variant
_pip(["uninstall", "-y",
      "onnxruntime", "onnxruntime-gpu", "onnxruntime-training",
      "rapidocr", "rapidocr_paddle", "rapidocr_onnxruntime", "rapidocr_openvino"])

# 2. Install CUDA-12-compatible onnxruntime-gpu 1.19.2 + pinned Pillow
_pip(["install",
      "pillow==10.4.0",
      "onnxruntime-gpu==1.19.2",   # ← CUDA 12.x, cuDNN 9, forgiving conv planner
      "rapidocr",
      "pymupdf",
      "tqdm"])

# 3. Downgrade onnx so the verification test model generates IR v10
_pip(["install", "onnx==1.16.1"])

print("\n" + "=" * 64)
print("✅ Installed onnxruntime-gpu 1.19.2 + onnx 1.16.1")
print("➡  NOW:  Run  →  Restart & Clear Cell Outputs")
print("=" * 64)

In [6]:
import PIL, onnxruntime as ort, numpy as np

print("Pillow      :", PIL.__version__)
print("ORT version :", ort.__version__)
print("Registered  :", ort.get_available_providers())

# ── Force a real CUDA session to prove the provider works ───────────────
import onnx
from onnx import helper, TensorProto

node  = helper.make_node("Add", inputs=["a", "b"], outputs=["c"])
graph = helper.make_graph(
    [node], "g",
    [helper.make_tensor_value_info("a", TensorProto.FLOAT, [4]),
     helper.make_tensor_value_info("b", TensorProto.FLOAT, [4])],
    [helper.make_tensor_value_info("c", TensorProto.FLOAT, [4])],
)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])

try:
    sess = ort.InferenceSession(
        model.SerializeToString(),
        providers=["CUDAExecutionProvider"]
    )
    out = sess.run(None, {"a": np.ones(4, np.float32),
                          "b": np.ones(4, np.float32)})
    print("✅ CUDA session OK — output:", out[0])
    print("   Active providers:", sess.get_providers())
except Exception as e:
    print("❌ CUDA session FAILED:", e)
    print("   → Try the next version down: 1.19.2, then 1.18.1, then 1.17.1")

Pillow      : 10.4.0
ORT version : 1.19.2
Registered  : ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
✅ CUDA session OK — output: [2. 2. 2. 2.]
   Active providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']


In [8]:
# ══════════════════════════════════════════════════════════════════════════
#  SCANNED PDF OCR — DUAL T4 via SUBPROCESS (fork-safe, notebook-safe)
#  Writes a standalone worker script, launches one fresh interpreter per GPU.
# ══════════════════════════════════════════════════════════════════════════
import os, sys, json, time, zipfile, subprocess
from pathlib import Path

# ── 1. Locate source ───────────────────────────────────────────────────
def _find_source_dir():
    for name in ("scanned_pdfs", "scanned-pdfs"):
        for p in Path("/kaggle/input").rglob(name):
            if p.is_dir():
                return p
    for p in Path("/kaggle/input").rglob("*"):
        if p.is_dir() and any(p.rglob("*.pdf")):
            return p
    return None

SOURCE_DIR = _find_source_dir()
assert SOURCE_DIR, "No scanned PDFs found under /kaggle/input"
print(f"✓ SOURCE_DIR = {SOURCE_DIR}")

OUTPUT_DIR    = Path("/kaggle/working/extracted_scanned")
ZIP_PATH      = Path("/kaggle/working/scanned_text.zip")
WORKER_SCRIPT = Path("/kaggle/working/_ocr_worker.py")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── 2. Write standalone worker (a REAL .py file, not a notebook fn) ────
WORKER_CODE = r'''
import os, sys, io, json, time
from pathlib import Path

# ─── CRITICAL ORDER ────────────────────────────────────────────────────
# 1) Read GPU id and set CUDA_VISIBLE_DEVICES *before* any CUDA-aware import
gpu_id = int(sys.argv[1])
os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_id)

# 2) Only now import CUDA-aware libraries
import pymupdf
import numpy as np
from PIL import Image
from rapidocr import RapidOCR, EngineType
import logging
logging.getLogger("RapidOCR").setLevel(logging.ERROR)

# 3) Args
pdf_list_file = Path(sys.argv[2])
output_dir    = Path(sys.argv[3])
source_dir    = Path(sys.argv[4])
progress_file = Path(sys.argv[5])

pdf_paths = [Path(l.strip()) for l in pdf_list_file.read_text().splitlines() if l.strip()]

# 4) Build engine (T4-friendly)
engine = RapidOCR(params={
    "Det.engine_type": EngineType.ONNXRUNTIME,
    "Cls.engine_type": EngineType.ONNXRUNTIME,
    "Rec.engine_type": EngineType.ONNXRUNTIME,
    "EngineConfig.onnxruntime.use_cuda": True,
    "EngineConfig.onnxruntime.cudnn_conv_algo_search": "HEURISTIC",
    "EngineConfig.onnxruntime.cudnn_conv_use_max_workspace": "0",
})
_ = engine(np.zeros((64, 64, 3), dtype=np.uint8))          # warm-up

success = failed = total_pages = total_blank = 0

for idx, pdf_path in enumerate(pdf_paths, 1):
    try:
        doc   = pymupdf.open(str(pdf_path))
        n_pg  = len(doc)
        if n_pg == 0:
            doc.close()
            continue

        parts, n_blank = [], 0
        for p in range(n_pg):
            pix = doc[p].get_pixmap(dpi=250)
            arr = np.array(Image.open(io.BytesIO(pix.tobytes("png"))))
            if arr.ndim == 3:
                arr = arr[:, :, ::-1]              # RGB → BGR
            res  = engine(arr)
            txts = getattr(res, "txts", None)
            txt  = "\n".join(txts) if txts else None
            del arr
            if txt and len(txt.strip()) >= 3:
                parts.append(f"--- Page {p+1} ---\n{txt}")
            else:
                n_blank += 1

        doc.close()
        total_pages += n_pg
        total_blank += n_blank

        rel     = pdf_path.relative_to(source_dir)
        out_pth = (output_dir / rel).with_suffix(".txt")
        out_pth.parent.mkdir(parents=True, exist_ok=True)
        out_pth.write_text("\n\n".join(parts) if parts else "", encoding="utf-8")
        success += 1

    except Exception as e:
        failed += 1
        print(f"[GPU{gpu_id}] ERROR {pdf_path.name}: {e}", flush=True)

    progress_file.write_text(json.dumps({
        "gpu": gpu_id, "done": idx, "total": len(pdf_paths),
        "success": success, "failed": failed,
        "pages": total_pages, "blank": total_blank,
    }))

print(f"[GPU{gpu_id}] DONE  ok={success}  fail={failed}  "
      f"pages={total_pages}  blank={total_blank}", flush=True)
'''

WORKER_SCRIPT.write_text(WORKER_CODE)
print(f"✓ Worker script → {WORKER_SCRIPT}")

# ── 3. Split PDFs into two halves ──────────────────────────────────────
pdf_files = sorted(SOURCE_DIR.rglob("*.pdf"))
print(f"✓ Found {len(pdf_files)} scanned PDFs")
assert pdf_files, "No PDFs found"

mid    = len(pdf_files) // 2
chunks = [pdf_files[:mid], pdf_files[mid:]]
print(f"→ GPU 0: {len(chunks[0])} PDFs  |  GPU 1: {len(chunks[1])} PDFs")

# ── 4. Launch one subprocess per GPU ───────────────────────────────────
procs, progress_files, log_files = [], [], []

for gpu_id, chunk in enumerate(chunks):
    list_file     = Path(f"/kaggle/working/_pdf_list_gpu{gpu_id}.txt")
    progress_file = Path(f"/kaggle/working/_progress_gpu{gpu_id}.json")
    log_file      = Path(f"/kaggle/working/_log_gpu{gpu_id}.txt")

    list_file.write_text("\n".join(str(p) for p in chunk))

    cmd = [sys.executable, str(WORKER_SCRIPT),
           str(gpu_id), str(list_file), str(OUTPUT_DIR),
           str(SOURCE_DIR), str(progress_file)]

    # Redirect child stdout+stderr to a log file to avoid pipe-buffer deadlock
    f = open(log_file, "w")
    p = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT)

    procs.append((gpu_id, p, f))
    progress_files.append(progress_file)
    log_files.append(log_file)
    print(f"→ GPU {gpu_id} worker launched  (PID {p.pid}, log → {log_file.name})")

# ── 5. Monitor progress via the per-GPU JSON files ─────────────────────
t0, last_done = time.time(), 0
while any(p.poll() is None for _, p, _ in procs):
    time.sleep(5)
    done_total = 0
    for pf in progress_files:
        if pf.exists():
            try:
                done_total += json.loads(pf.read_text()).get("done", 0)
            except Exception:
                pass
    if done_total > last_done:
        el = time.time() - t0
        print(f"   {done_total}/{len(pdf_files)}  "
              f"({el:.0f}s, {el/max(done_total,1):.2f} s/pdf)")
        last_done = done_total

# Close all log file handles
for _, p, f in procs:
    p.wait()
    f.close()

elapsed = time.time() - t0

# ── 6. Aggregate results ───────────────────────────────────────────────
agg = {"ok": 0, "fail": 0, "pages": 0, "blank": 0}
for pf in progress_files:
    if pf.exists():
        d = json.loads(pf.read_text())
        agg["ok"]    += d.get("success", 0)
        agg["fail"]  += d.get("failed", 0)
        agg["pages"] += d.get("pages", 0)
        agg["blank"] += d.get("blank", 0)

print(f"\n{'='*60}")
print(f"DONE  ✓ success={agg['ok']}   ✗ failed={agg['fail']}")
print(f"Pages OCR'd : {agg['pages']}  (blank {agg['blank']})")
print(f"Time        : {elapsed/60:.2f} min  "
      f"({elapsed/max(agg['pages'],1)*1000:.0f} ms/page)")

print("\n--- Per-GPU tail ---")
for lf in log_files:
    if lf.exists():
        lines = lf.read_text().splitlines()[-8:]
        print(f"  {lf.name}:")
        for l in lines:
            print(f"    {l}")

# ── 7. Zip ─────────────────────────────────────────────────────────────
print("\n→ Creating zip…")
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, "w",
                    zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for fp in OUTPUT_DIR.rglob("*.txt"):
        zf.write(fp, fp.relative_to(OUTPUT_DIR))

print(f"✓ Zip: {ZIP_PATH}  ({ZIP_PATH.stat().st_size/1e6:.2f} MB)")

print("\n→ /kaggle/working/ contents:")
for p in sorted(Path("/kaggle/working").iterdir()):
    if p.is_file():
        print(f"   {p.name:34s} {p.stat().st_size/1e6:8.2f} MB")
    else:
        n = sum(1 for _ in p.rglob("*") if _.is_file())
        print(f"   {p.name + '/':34s} {n:>6} files")

print("\n✅ DONE — download scanned_text.zip from the right-sidebar 'Output' tab.")

✓ SOURCE_DIR = /kaggle/input/datasets/fayaz042/scanned-pdfs
✓ Worker script → /kaggle/working/_ocr_worker.py
✓ Found 196 scanned PDFs
→ GPU 0: 98 PDFs  |  GPU 1: 98 PDFs
→ GPU 0 worker launched  (PID 954, log → _log_gpu0.txt)
→ GPU 1 worker launched  (PID 955, log → _log_gpu1.txt)
   1/196  (35s, 35.00 s/pdf)
   2/196  (40s, 20.00 s/pdf)
   3/196  (55s, 18.33 s/pdf)
   4/196  (70s, 17.50 s/pdf)
   5/196  (75s, 15.00 s/pdf)
   6/196  (90s, 15.00 s/pdf)
   7/196  (105s, 15.00 s/pdf)
   8/196  (115s, 14.38 s/pdf)
   9/196  (120s, 13.33 s/pdf)
   11/196  (140s, 12.73 s/pdf)
   13/196  (155s, 11.92 s/pdf)
   14/196  (165s, 11.79 s/pdf)
   15/196  (170s, 11.33 s/pdf)
   16/196  (175s, 10.94 s/pdf)
   17/196  (185s, 10.88 s/pdf)
   18/196  (190s, 10.56 s/pdf)
   19/196  (195s, 10.26 s/pdf)
   20/196  (205s, 10.25 s/pdf)
   22/196  (215s, 9.77 s/pdf)
   23/196  (225s, 9.78 s/pdf)
   24/196  (240s, 10.00 s/pdf)
   25/196  (255s, 10.20 s/pdf)
   26/196  (260s, 10.00 s/pdf)
   27/196  (270s, 10.0

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  ZIP the extracted folder → /kaggle/working/scanned_text.zip
#  Then download from right-sidebar → "Output" tab → ⋮ → Download
# ══════════════════════════════════════════════════════════════════════════
import os, zipfile
from pathlib import Path

EXTRACTED_DIR = Path("/kaggle/working/extracted_scanned")
ZIP_PATH      = Path("/kaggle/working/scanned_text.zip")

# ── Safety checks ──────────────────────────────────────────────────────
assert EXTRACTED_DIR.exists(), f"❌ {EXTRACTED_DIR} not found — run the OCR pipeline first."

txt_files = list(EXTRACTED_DIR.rglob("*.txt"))
assert txt_files, f"❌ No .txt files inside {EXTRACTED_DIR}."
print(f"→ Found {len(txt_files)} .txt files")

# ── Remove previous zip if any ─────────────────────────────────────────
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

# ── Create the zip ─────────────────────────────────────────────────────
print(f"→ Writing {ZIP_PATH.name} …")
with zipfile.ZipFile(ZIP_PATH, "w",
                    compression=zipfile.ZIP_DEFLATED,
                    compresslevel=6) as zf:
    for i, fp in enumerate(txt_files, 1):
        zf.write(fp, fp.relative_to(EXTRACTED_DIR))
        if i % 200 == 0:
            print(f"   {i}/{len(txt_files)} …")

# ── Verify integrity ───────────────────────────────────────────────────
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    assert zf.testzip() is None, "❌ Corrupt entry found!"
    n_entries = len(zf.namelist())

# ── Report ─────────────────────────────────────────────────────────────
src_mb = sum(f.stat().st_size for f in txt_files) / 1e6
zip_mb = ZIP_PATH.stat().st_size / 1e6
print(f"\n✓ Zip verified : {n_entries} entries, no corruption")
print(f"  extracted_scanned/  : {src_mb:.2f} MB")
print(f"  {ZIP_PATH.name:20s}: {zip_mb:.2f} MB  "
      f"(compression {100*(1 - zip_mb/max(src_mb,1e-9)):.0f}%)")

print("\n→ /kaggle/working/ contents:")
for p in sorted(Path("/kaggle/working").iterdir()):
    if p.is_file():
        print(f"   {p.name:34s} {p.stat().st_size/1e6:8.2f} MB")
    else:
        n = sum(1 for _ in p.rglob("*") if _.is_file())
        print(f"   {p.name + '/':34s} {n:>6} files")

print(f"\n✅ DONE — open the right-sidebar 'Output' tab, find {ZIP_PATH.name},")
print("   click ⋮  →  Download.")